# Tasks & Expected Outputs

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) CrewAI roadmap.

You will learn how to define CrewAI tasks, use `output_file`, and inspect `TaskOutput` after a run.

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 1. Basic task creation

A task needs `description` and `expected_output`. Assign an `agent` so the crew knows who executes it.

In [ ]:
from crewai import Agent, Task, Crew

researcher = Agent(
    role="Research Analyst",
    goal="Summarize topics clearly and accurately",
    backstory="You are a careful researcher who cites facts and avoids speculation.",
    verbose=True,
)

summarize_task = Task(
    description="Summarize the benefits of multi-agent systems for product teams in 3–5 bullet points.",
    expected_output="A short bullet list; each bullet one sentence; no introduction paragraph.",
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[summarize_task], verbose=True)
result = crew.kickoff()
print(result)

## 2. Task with `output_file`

Persist the task result to disk. Create the directory first if it does not exist.

In [ ]:
import pathlib

from crewai import Agent, Task, Crew

pathlib.Path("outputs").mkdir(parents=True, exist_ok=True)

writer = Agent(
    role="Technical Writer",
    goal="Produce clean written deliverables",
    backstory="You write concise technical prose suitable for internal docs.",
    verbose=True,
)

doc_task = Task(
    description="Write a one-paragraph overview of CrewAI tasks and expected outputs.",
    expected_output="A single paragraph, 4–6 sentences, suitable for a README.",
    agent=writer,
    output_file="outputs/crewai_tasks_overview.md",
)

crew = Crew(agents=[writer], tasks=[doc_task], verbose=True)
crew.kickoff()

## 3. Accessing `TaskOutput` properties

After `kickoff()`, inspect `.raw` and `.agent` on the task's `output`. (`pydantic` / `json_dict` apply when you configure structured output.)

In [ ]:
from crewai import Agent, Task, Crew

analyst = Agent(
    role="Analyst",
    goal="Answer in one short sentence",
    backstory="You give minimal, direct answers.",
    verbose=True,
)

quick_task = Task(
    description="In one sentence, what is a CrewAI Task?",
    expected_output="One sentence only.",
    agent=analyst,
)

crew = Crew(agents=[analyst], tasks=[quick_task], verbose=True)
crew.kickoff()

print("raw:", quick_task.output.raw)
print("agent:", quick_task.output.agent)

## Key takeaways

- Tasks pair `description` (what to do) with `expected_output` (what the deliverable must look like).
- `output_file` writes the result to a path you choose; create parent folders if needed.
- After execution, `task.output` exposes `.raw`, `.pydantic`, `.json_dict`, and `.agent` for programmatic use.
- Specific `expected_output` text improves consistency, especially when tasks chain via `context`.